# Traffic sign detection with YOLOv8n

Training and evaluation of YOLOv8n on the robot-captured dataset. Run the cells from top to bottom on a GPU runtime.

1. Setup
2. Dataset
3. Training
4. Evaluation
5. Export

## 1. Setup

In [ ]:
!pip install -q ultralytics roboflow

import torch, ultralytics
print("ultralytics:", ultralytics.__version__)
print("torch      :", torch.__version__)
print("GPU        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
from pathlib import Path
from getpass import getpass

# Class order used in the dataset and in all evaluations
CLASSES = [
    "R-25a_esquerda",
    "R-25b_direita",
    "R-26_frente",
    "R-3_sentido-proibido",
    "R-4a_proibido-esquerda",
    "R-4b_proibido-direita",
]

# Any Roboflow account key works, since the dataset is public
RF_KEY = getpass("Roboflow API key: ")

OUT = Path("/content/export")
OUT.mkdir(exist_ok=True)

In [ ]:
# Reference video
import requests

file_id = "1lEnlrQsOOebodOIKd7LiijreMVIVnNp_"
url = f"https://drive.google.com/uc?export=download&id={file_id}"

r = requests.get(url, stream=True)
with open("/content/reference_video.mp4", "wb") as f:
    for chunk in r.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

!ls -lh /content/reference_video.mp4

## 2. Dataset

Images captured by the ESP32-CAM in five sessions, split by session: sessions 1 to 3 for training, session 4 for validation and session 5 for testing.

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key=RF_KEY)
ds = rf.workspace("vitor-chagas").project("robocar-dataset-c").version(2).download("yolov8")
ROOT = Path(ds.location)
print(ROOT)

In [ ]:
# Images and annotations per split and class
from collections import Counter
import yaml

names = yaml.safe_load(open(ROOT / "data.yaml"))["names"]
for split in ["train", "valid", "test"]:
    if not (ROOT / split / "images").exists():
        continue
    count = Counter()
    n_images = n_background = 0
    # Iterate over images, not labels, because background images may have no label file
    for img in (ROOT / split / "images").iterdir():
        n_images += 1
        txt = ROOT / split / "labels" / (img.stem + ".txt")
        lines = txt.read_text().splitlines() if txt.exists() else []
        lines = [l for l in lines if l.strip()]
        if not lines:
            n_background += 1
        for line in lines:
            count[int(line.split()[0])] += 1
    print(f"\n--- {split} ---")
    print(f"images            : {n_images}")
    print(f"background images : {n_background}")
    print(f"annotations       : {sum(count.values())}")
    for i, name in enumerate(names):
        print(f"  {count[i]:5d}  {name}")

In [ ]:
# The class order in data.yaml must match CLASSES
print("dataset:", list(names))
print("CLASSES:", CLASSES)
assert list(names) == CLASSES, "class order does not match"

## 3. Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # COCO pretrained weights

model.train(
    data=str(ROOT / "data.yaml"),
    epochs=150,
    imgsz=640,
    batch=16,
    seed=0,
    fliplr=0.0,    # mirroring would swap left and right signs
    degrees=8.0,
    hsv_v=0.4,
    hsv_s=0.5,
    patience=30,
    project="/content/runs",
    name="A_v8n",
)

## 4. Evaluation

In [ ]:
# Test set (session 5, not used in training)
best = YOLO("/content/runs/A_v8n/weights/best.pt")
val = best.val(data=str(ROOT / "data.yaml"), split="test",
               project="/content/runs", name="A_val_test")

print(f"mAP@0.5      : {val.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {val.box.map:.4f}")
print(f"precision    : {val.box.mp:.4f}")
print(f"recall       : {val.box.mr:.4f}")
print("\nper class:")
for i, name in enumerate(CLASSES):
    p, r, _, _ = val.box.class_result(i)
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    print(f"  {name:26s} P={p:.3f}  R={r:.3f}  F1={f1:.3f}  mAP50={val.box.ap50[i]:.3f}")

In [ ]:
# Reference video. stream=True processes one frame at a time and avoids
# running out of memory on Colab; the generator must be consumed to run.
results = best.predict(
    source="/content/reference_video.mp4",
    conf=0.5,
    save=True,
    stream=True,
    project="/content/runs",
    name="A_video",
)
for _ in results:
    pass

## 5. Export

In [ ]:
# Copy weights, metrics and figures of each run to /content/export
import shutil

FILES = ["weights/best.pt", "args.yaml", "results.csv", "results.png",
         "confusion_matrix.png", "confusion_matrix_normalized.png",
         "BoxPR_curve.png", "BoxF1_curve.png", "BoxP_curve.png", "BoxR_curve.png",
         "val_batch0_pred.jpg", "val_batch0_labels.jpg",
         "val_batch1_pred.jpg", "val_batch1_labels.jpg",
         "val_batch2_pred.jpg", "val_batch2_labels.jpg"]

for run in ["A_v8n", "A_val_test"]:
    src = Path("/content/runs") / run
    dst = OUT / run
    dst.mkdir(parents=True, exist_ok=True)
    for rel in FILES:
        if (src / rel).exists():
            shutil.copy(src / rel, dst / Path(rel).name)

# data.yaml identifies the dataset version used in the run
shutil.copy(ROOT / "data.yaml", OUT / "data.yaml")

for video in (Path("/content/runs") / "A_video").glob("*.avi"):
    shutil.copy(video, OUT / "A_video.avi")

!find /content/export -type f | sort

In [ ]:
from google.colab import files
shutil.make_archive("/content/export", "zip", OUT)
files.download("/content/export.zip")